In [ ]:
import pyarrow.parquet as pq
import glob
import os

# ==================================================
# ✅ 配置区
# ==================================================
INPUT_PATTERN = "yellow_tripdata_2024-*.parquet"
OUTPUT_FILE = "yellow_tripdata_2024_combined.parquet"
# ==================================================

print("1️⃣ 查找 2024 年 yellow taxi 文件...")
files = sorted(glob.glob(INPUT_PATTERN))

if not files:
    raise FileNotFoundError("没有找到 yellow_tripdata_2024-*.parquet 文件")

print(f"   找到 {len(files)} 个月份文件")

print("2️⃣ 合并 parquet（不破坏结构）...")

writer = None

for f in files:
    pf = pq.ParquetFile(f)

    if writer is None:
        # ✅ 正确获取 Arrow Schema
        schema = pf.schema_arrow
        writer = pq.ParquetWriter(OUTPUT_FILE, schema)

    for batch in pf.iter_batches():
        writer.write_batch(batch)

writer.close()

print("✅ 合并完成")
print("输出文件:", OUTPUT_FILE)

In [4]:
import pyarrow.parquet as pq
import pyarrow as pa
import glob
import os

# ==================================================
# ✅ 配置区
# ==================================================
INPUT_PARQUET = "yellow_tripdata_2024_combined.parquet"
OUTPUT_PARQUET = "yellow_tripdata_2024_combined_with_type.parquet"
NEW_COLUMN = "trip_type"
NEW_VALUE = "yellow_taxi"
# ==================================================

print("1️⃣ 打开合并后的 yellow taxi 文件...")
pf = pq.ParquetFile(INPUT_PARQUET)

schema = pf.schema_arrow
writer = None

print("2️⃣ 逐 batch 添加 trip_type 列...")

for batch in pf.iter_batches():
    df = batch.to_pandas()

    # ✅ 添加新列（所有行值相同）
    df[NEW_COLUMN] = NEW_VALUE

    # 转回 Arrow Table
    table = pa.Table.from_pandas(df, schema=schema)

    if writer is None:
        # ✅ 初始化 writer（包含新列）
        writer = pq.ParquetWriter(OUTPUT_PARQUET, table.schema)

    writer.write_batch(table)

writer.close()

print("✅ 完成")
print("输出文件:", OUTPUT_PARQUET)
print("新增列:", NEW_COLUMN)

1️⃣ 打开合并后的 yellow taxi 文件...
2️⃣ 逐 batch 添加 trip_type 列...


TypeError: Cannot convert pyarrow.lib.Table to pyarrow.lib.RecordBatch